# Notebook 08 -- embed_one_encoder

**What this notebook is for.** Every answer in the corpus is turned into an
*embedding*: a long list of numbers standing for the meaning of the passage, so
that two answers making the same point land close together even when they share no
words. This notebook produces those numbers with ONE encoder and writes them to
disk. Notebook 09 is where they become per-facet similarity scores.

**Run it once per encoder.** One line below, `ENCODER_KEY`, picks which one:

| key | model | role |
| --- | --- | --- |
| `bge` | BAAI/bge-large-en-v1.5 | REQUIRED, the primary encoder |
| `mpnet` | sentence-transformers/all-mpnet-base-v2 | optional, continuity with the earlier Dark Side pipeline |
| `voyage` | voyage-3-large | optional, hosted API, needs a key in Colab Secrets |

Run `bge` first; it is the leg the pipeline requires. The other two are the
model-sensitivity report: prereg 6.7 asks for more than one embedding model, and
MEASUREMENT_DESIGN.md section 5 pins three from three different makers so that a
quirk shared by one maker's models cannot present itself as cross-model agreement.

**This is the GPU notebook.** Before running anything: Runtime > Change runtime
type > T4 GPU. A GPU is a speed matter and not a correctness one -- the same
checkpoint on a CPU returns the same vectors, just far more slowly -- but on CPU a
full encode is a long wait instead of a couple of minutes. The Voyage path runs on
somebody else's servers and needs no accelerator at all.

**Reads:** `openends_long.csv` (notebook 00) and `embedding_seed_pools.json`
(notebook 04). **Writes:** three files, which is what notebook 09 reads --
`embeddings_<key>.npy`, one row per answer; `embeddings_<key>_seed.npy`, one row
per reference passage, embedded by the same checkpoint in the same run; and
`embeddings_<key>_meta.json`, which says how many rows are in each, how wide they
are, which answer each row of the first is and which facet each row of the second
belongs to.

**What this notebook cannot do.** Nothing here can pass or fail a facet. The
registered gate has two clauses -- the models agree with each other at an
intraclass correlation (ICC, a measure of how closely independent scorers track the
same responses) of at least .60, and each facet correlates at least .30 with its
matching survey scale -- and neither is computed from an embedding. Similarity is
secondary convergence: reported and inspected, never folded into a weighted
composite, because weighting each stream by its agreement would let an unstable
embedding stream quietly move a headline number.

## Config

In [47]:
# === CONFIG: the only cell you edit ===
# Every switch in the pipeline lives here. Nothing below this cell needs changing.

RUN_MODE    = "full"        # "smoke" = first 20 rows, costs pennies, proves the
                            #           whole chain works end to end.
                            # "full"  = the real corpus.
HUMAN_GOLD  = False         # False = the pre-registered pipeline. Language models
                            #         do the coding, exactly as prereg 6.7 says.
                            #         This is the default and it runs start to
                            #         finish with no human step.
                            # True  = ALSO run the human-coder arm. That arm is a
                            #         DEVIATION from the registration and needs the
                            #         OSF addendum posted first.
WORKDIR     = "/content/retribution_nlp"   # where every file in the chain lives
RANDOM_SEED = 20260629      # frozen. Changing it re-draws every random split.

# --- PER-NOTEBOOK: the three lines below differ in each notebook ---
NB_ID       = "08"                     # this notebook's number
NB_NAME     = "embed_one_encoder"      # this notebook's job, in two words
NB_ASKS     = "Turn every answer into an embedding with ONE encoder. Run once per encoder."

## Setup

In [48]:
# === SETUP: folders, the manifest, and the two functions every notebook uses ===
# WHAT THIS CELL DOES: makes the working folder, loads the manifest, and defines
# need() and done(). You never edit this cell.
#
# THE MANIFEST is a small file called _manifest.json that lives beside the data.
# Every notebook records what it produced in there. The next notebook reads it to
# check its inputs exist. That is what lets a notebook tell you "run 03 first"
# instead of dying on a missing file three cells later.

import os, json, hashlib, datetime, sys

os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
MANIFEST = os.path.join(WORKDIR, "_manifest.json")

def _load_manifest():
    """Read the manifest, or start an empty one on the first notebook."""
    if os.path.exists(MANIFEST):
        with open(MANIFEST) as f:
            return json.load(f)
    return {"files": {}, "runs": []}

def _sha256(path, cap=8_000_000):
    """First 8 MB is plenty to catch a file that changed underneath us."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        h.update(f.read(cap))
    return h.hexdigest()[:16]

class MissingInput(Exception):
    """Raised when a required upstream file is not there yet."""
    pass

def need(filename, produced_by=None, optional=False):
    """Return the path to an input file, or explain which notebook makes it.

    This is the function that makes the chain navigable. A missing input stops
    the notebook HERE, with a sentence telling you what to run, rather than
    surfacing as a confusing error further down.
    """
    path = os.path.join(WORKDIR, filename)
    if os.path.exists(path):
        rec = _load_manifest()["files"].get(filename, {})
        stamp = rec.get("written_at", "unknown time")
        print(f"  [ok]   {filename}  (written {stamp})")
        return path
    if optional:
        print(f"  [skip] {filename} is not here. This notebook will carry on "
              f"without it and say so in its output.")
        return None
    where = f" Run notebook {produced_by} first." if produced_by else ""
    raise MissingInput(
        f"\n\n  MISSING INPUT: {filename}\n"
        f"  This notebook cannot run without it.{where}\n"
        f"  Files present in {WORKDIR}:\n    "
        + "\n    ".join(sorted(os.listdir(WORKDIR)) or ["(none)"]) + "\n")

def done(filename, rows=None, note=""):
    """Record a file this notebook produced, so later notebooks can find it."""
    path = os.path.join(WORKDIR, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(f"done() called for {filename} but it is not there")
    m = _load_manifest()
    m["files"][filename] = {
        "written_by": f"{NB_ID}_{NB_NAME}",
        "written_at": datetime.datetime.now().isoformat(timespec="seconds"),
        "rows": rows,
        "sha16": _sha256(path),
        "run_mode": RUN_MODE,
        "note": note,
    }
    m["runs"] = [r for r in m["runs"] if r.get("nb") != NB_ID]
    m["runs"].append({"nb": NB_ID, "name": NB_NAME, "run_mode": RUN_MODE,
                      "human_gold": HUMAN_GOLD,
                      "at": datetime.datetime.now().isoformat(timespec="seconds")})
    with open(MANIFEST, "w") as f:
        json.dump(m, f, indent=1)
    size = os.path.getsize(path)
    print(f"  [wrote] {filename}  ({rows if rows is not None else '?'} rows, "
          f"{size:,} bytes)")

print(f"NOTEBOOK {NB_ID} -- {NB_NAME}")
print(f"  {NB_ASKS}")
print(f"  mode: RUN_MODE={RUN_MODE}, HUMAN_GOLD={HUMAN_GOLD}")
print(f"  folder: {WORKDIR}")
_m = _load_manifest()
if _m["runs"]:
    print("  already run in this folder: "
          + ", ".join(sorted(r["nb"] for r in _m["runs"])))
else:
    print("  this folder is empty: notebook 00 is the place to start.")

NOTEBOOK 08 -- embed_one_encoder
  Turn every answer into an embedding with ONE encoder. Run once per encoder.
  mode: RUN_MODE=full, HUMAN_GOLD=False
  folder: /content/retribution_nlp
  already run in this folder: 08


## Load

In [49]:
# === PICK THE ENCODER: the one line in this notebook you edit ===
# WHAT THIS CELL DOES: chooses which encoder this run uses. Everything below reads
# the chosen entry and never names a model again, so the three runs of this
# notebook differ in exactly one line and cannot drift apart through a copy-edit.
#
# WHY MORE THAN ONE ENCODER: prereg 6.7 asks for "similarity to short reference
# passages for each facet, computed with more than one embedding model".
# MEASUREMENT_DESIGN.md section 5 pins three, from three different makers, so a
# quirk shared by one maker's models cannot present itself as cross-model
# agreement. The three runs are what notebook 09's model-sensitivity table needs:
# the ABSOLUTE similarities are expected to shift between encoders (that shift is
# the reported finding, not a defect), while the RANK order of responses is
# expected to hold.
#
# WHY NONE OF THEM CAN PASS OR FAIL A FACET: the registered gate has exactly two
# clauses -- inter-model ICC of at least .60, and a correlation of at least .30
# with the matching survey scale -- and neither is computed from an embedding.

ENCODER_KEY = "voyage"    # "bge"    REQUIRED, primary, open weights, runs locally
                       # "mpnet"  optional, the encoder the earlier Dark Side
                       #          (Punishment 2.1) pipeline used
                       # "voyage" optional, hosted API, needs a key in Secrets

In [50]:
# === THE THREE ENCODERS, PINNED ===
# WHY PINNED CHECKPOINT STRINGS AND NEVER A MOVING ALIAS: the checkpoint is the
# model half of reproducibility and the pip install below is the library half;
# neither alone is sufficient. An alias that silently rolled to a new generation
# would move every cosine in notebook 09 while the notebook text looked untouched.
ENCODERS = {
    # REQUIRED / primary. Open weights, so the whole embedding leg can be
    # reproduced without an API account or a paid key, and it comes from a third
    # maker. This is the leg notebook 09 reports as the encoder of record.
    "bge": dict(maker="BAAI", checkpoint="BAAI/bge-large-en-v1.5",
                kind="local", dim=1024, batch_size=32, required=True,
                size="~1.3 GB download"),
    # OPTIONAL. The encoder the prior Dark Side (Punishment 2.1) pipeline used,
    # carried here for continuity. It is also the model whose ABSOLUTE numbers
    # moved furthest there -- the "% closer to dark" metric read 67.5% on mpnet
    # against 21-28% on BGE and Voyage, while rank order held at r ~ .71 -- which
    # is exactly what makes it valuable in a sensitivity report and exactly why it
    # is not the required leg.
    "mpnet": dict(maker="sentence-transformers",
                  checkpoint="sentence-transformers/all-mpnet-base-v2",
                  kind="local", dim=768, batch_size=32, required=False,
                  size="~440 MB download"),
    # OPTIONAL. The commercial flagship of the three-maker set. Being an API it is
    # the one encoder that can fail for reasons having nothing to do with the data
    # (rate limit, transient 5xx, expired key), which is why it carries a retry
    # wrapper below and why BGE, which cannot fail that way, is the required leg.
    # Which Voyage generation to pin is team decision 1 in MEASUREMENT_DESIGN.md
    # section 11. If the pin moves, delete this encoder's .npy and re-run: a
    # half-old, half-new matrix is not a set of comparable vectors.
    "voyage": dict(maker="Voyage AI", checkpoint="voyage-3-large",
                   kind="api", dim=1024, batch_size=64, required=False,
                   size="hosted, no download"),
}
assert ENCODER_KEY in ENCODERS, f"ENCODER_KEY must be one of {sorted(ENCODERS)}"
ENC = ENCODERS[ENCODER_KEY]

# Every output name carries the encoder key, so three runs cannot overwrite each
# other and notebook 09 can tell at a glance which encoders have been run.
# THREE FILES PER ENCODER, and notebook 09 expects all three under exactly these
# names: the answers, the reference passages, and the index saying what every row
# of each one is. The two matrices are separate files because they are produced on
# different schedules -- the answers are cached, since a full encode costs GPU
# minutes or API spend, while the reference passages are re-embedded on every run,
# since notebook 04 can rebuild the pools at any time. They are nonetheless always
# WRITTEN together, by one checkpoint in one run, and notebook 09 re-checks that
# pairing rather than taking it on trust.
OUT_NPY  = f"embeddings_{ENCODER_KEY}.npy"        # one row per answer
OUT_SEED = f"embeddings_{ENCODER_KEY}_seed.npy"   # one row per reference passage
OUT_META = f"embeddings_{ENCODER_KEY}_meta.json"  # what each row of each one is

print(f"encoder   : {ENCODER_KEY}  ({ENC['maker']}, {ENC['checkpoint']})")
print(f"role      : {'REQUIRED primary' if ENC['required'] else 'optional'}"
      f"   ({ENC['size']})")
print(f"writes    : {OUT_NPY}, {OUT_SEED}, {OUT_META}")

encoder   : voyage  (Voyage AI, voyage-3-large)
role      : optional   (hosted, no download)
writes    : embeddings_voyage.npy, embeddings_voyage_seed.npy, embeddings_voyage_meta.json


In [51]:
# === INSTALL: only the one package this encoder needs ===
# WHAT: installs the package this run imports that a Colab VM does not ship, then
# records the versions the session actually resolved. It computes nothing.
#
# WHY IT RUNS EVERY TIME: a Colab VM is discarded when the runtime disconnects, so
# the install happens at the top of every session.
#
# WHY THE COMPILED CORE IS NEVER PINNED HERE: numpy, pandas and scipy ship in the
# Colab image compiled against one another. A pip install that resolves a different
# numpy leaves the already-built wheels pointing at the wrong C struct layout, and
# the next import dies with
#     ValueError: numpy.dtype size changed, may indicate binary incompatibility
# Nothing here needs a different numpy, so nothing asks for one. Reproducibility is
# served by RECORDING what the session resolved (printed below, and archived with
# the outputs) rather than by forcing a downgrade that stops the notebook running
# at all.
#
# IF YOU EVER SEE THAT ERROR: pip cannot swap a compiled module the live kernel has
# already imported. Runtime -> Restart session, then run from the top.
if ENC["kind"] == "local":
    !pip install -q sentence-transformers
else:
    !pip install -q voyageai

import importlib
from importlib import metadata as _md

def _ver(mod_name, dist_name=None):
    # Not every package exposes __version__, so fall back to the installed
    # distribution metadata before giving up.
    try:
        return importlib.import_module(mod_name).__version__
    except Exception:
        pass
    try:
        return _md.version(dist_name or mod_name)
    except Exception as e:
        return f"NOT AVAILABLE -- {type(e).__name__}"

print("\nResolved versions (archive these with the outputs):")
print(f"  python        {sys.version.split()[0]}")
for _mod in ["numpy", "pandas", "torch", "sentence_transformers", "voyageai"]:
    print(f"  {_mod:20s} " + str(_ver(_mod)))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 3.4 MB/s eta 0:00:00

Resolved versions (archive these with the outputs):
  python        3.13.15
  numpy                2.1.3
  pandas               2.2.3
  torch                2.11.0+cu128
  sentence_transformers 5.7.0
  voyageai             0.5.0


In [52]:
# === IMPORTS, THE FACET REGISTER, AND THE DEVICE ===
import time
import numpy as np
import pandas as pd
import warnings
# Encoder loads emit a wall of deprecation notices that would bury the printed
# diagnostics this notebook is actually read for, so those two categories are
# silenced BY NAME. Never a bare filterwarnings("ignore"): that silences every
# warning for the rest of the session, including numpy's RuntimeWarning on an
# invalid value or a divide by zero. That warning is the one thing that would tell
# a reader an embedding had come back degenerate, which is precisely what the row
# lengths printed at the end of this notebook exist to catch.
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

if ENC["kind"] == "local":
    # torch is imported for one purpose: to ask whether a GPU is attached, so the
    # checkpoint can be placed on it. GPU is a speed matter, not a correctness one:
    # the same checkpoint on a CPU returns the same vectors, just slowly.
    import torch
    from sentence_transformers import SentenceTransformer
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if device == "cuda":
        print(f"GPU available: {torch.cuda.get_device_name(0)}")
    else:
        print("No GPU attached. This will still run and will still be correct, but")
        print("it will be slow: Runtime > Change runtime type > T4 GPU, then rerun.")
else:
    import voyageai
    device = "api"   # the encoding happens on Voyage's servers; no accelerator here
    print("Voyage path: the encoding happens over the network, no GPU needed.")

# The nine facets, in the order fixed everywhere downstream.
# Six retributive facets -- four principled (desert, proportionality, censure,
# moral_balance) and two vengeful (revenge, suffering) -- plus the three
# consequentialist goals, which are scored so that retributive language is measured
# against a full menu of punishment justifications rather than in a vacuum
# (prereg 5.15). They are NEVER summed into a retributivism total, and there is no
# prosocial-minus-dark axis anywhere in this pipeline: the object of measurement is
# a nine-way distinction, not a valence.
# This ORDER is load-bearing: it fixes the facet order of every similarity frame
# downstream, so the same nine names in the same nine places must appear in every
# notebook that touches a facet.
FACETS = ["desert", "proportionality", "censure", "moral_balance",
          "revenge", "suffering", "deterrence", "incapacitation", "rehabilitation"]
print(f"facets: {len(FACETS)}")

Voyage path: the encoding happens over the network, no GPU needed.
facets: 9


In [53]:
# === LOAD 1 of 2: the corpus of answers ===
# One row per response_id x item_type. Every vector this notebook writes is
# positionally aligned to THIS frame, in THIS row order, so nothing below is
# allowed to sort, filter or drop a row after this cell.
print("Inputs:")
p_open = need("openends_long.csv", produced_by="00")
# keep_default_na=False, matching the way notebook 00 wrote this file and the way
# notebooks 01, 03 and 04 read it. A participant who literally answered "N/A" or
# "n/a" keeps that text instead of arriving as a missing value; two answers in this
# corpus do exactly that. Read without the flag they would be blanked here and
# embedded as empty strings, and the blank count printed below would disagree with
# the word-list leg's count of blanks in the very same file.
df = pd.read_csv(p_open, keep_default_na=False)

# Long-shape contract. The assert fails here, naming the missing column, rather
# than surfacing 200 lines later as an all-NaN similarity column in notebook 09.
assert {"response_id", "item_type", "text"}.issubset(df.columns), \
    "openends_long.csv must have response_id, item_type, text"
# Belt and braces on top of keep_default_na=False: `text` has to be a string,
# because the blank count in the next cell calls .str.strip() on it, and a column
# pandas typed as a number -- which is what it does when every value in the file is
# empty -- would raise there instead of reporting the emptiness.
df["text"] = df["text"].fillna("").astype(str)

# The three open-ends are scored SEPARATELY and never pooled (prereg 5.15): the
# sentence justification and the ranking explanation look BACKWARD, where desert
# and proportionality language sits, while the accomplishment item looks FORWARD,
# where deterrence / incapacitation / rehabilitation language sits. Combining them
# would mix the two registers whose contrast is itself a measure (E6).
# EXACT SET EQUALITY, not a subset test, matching notebooks 00, 01 and 06. An EXTRA
# value would silently create a fourth reference set inside notebook 09. A MISSING
# value means a whole open-end went missing: drop the forward item and this
# notebook embeds two thirds of the corpus without a murmur, taking the
# forward-versus-backward contrast E6 exists to measure with it.
assert set(df.item_type.unique()) == {"ranking", "backward", "forward"}, \
    f"Unexpected item_type values: {sorted(df.item_type.unique())}"

# ONE ROW PER (response_id, item_type). This is the contract the whole chain rests
# on, and it is checked HERE, on the file this notebook actually reads, rather than
# assumed to have been settled upstream. A duplicated pair would be embedded twice,
# would appear twice in the row keys saved beside the matrix, and would reach
# notebook 09 as two identical similarity rows for one person -- so every count
# downstream reads high while the printed participant total still says 948.
_dups = df.loc[df.duplicated(["response_id", "item_type"], keep=False),
               ["response_id", "item_type"]]
assert _dups.empty, (
    f"openends_long.csv has {len(_dups)} rows sharing a (response_id, item_type) "
    f"key, for example {_dups.head(3).to_dict('records')}. One row per answer is "
    f"what every alignment check here and in notebook 09 depends on.")

Inputs:
  [ok]   openends_long.csv  (written unknown time)


In [54]:
# === WHICH ROWS THIS RUN ENCODES, AND WHAT IS BLANK IN THEM ===
# Smoke mode is a plumbing test and never a result: it proves the chain runs end to
# end in about a minute. RUN_MODE travels into the metadata file, so a smoke matrix
# can never be mistaken for a full one downstream.
if RUN_MODE == "smoke":
    # Trim by PARTICIPANT, never by row. Every person contributes three rows, one
    # per open-ended question, and notebooks 00 and 03 both cut on participants.
    # A head(20) here would keep 20 ROWS, which is six and two thirds of a person,
    # and the three-rows-per-person check downstream would fail on the participant
    # left half in and half out.
    _keep = pd.unique(df["response_id"])[:20]
    df = df[df["response_id"].isin(_keep)].reset_index(drop=True)
    print(f"  SMOKE MODE: trimmed to {len(_keep)} participants, {len(df)} rows "
          f"(three answers each). A plumbing test, not a result.")

# fillna("") keeps the row rather than dropping it. Dropping blanks here would
# break the positional alignment between this list, df, and the saved matrix -- and
# that alignment is exactly what the row-count and row-key checks below protect. A
# blank encodes to a valid vector whose similarity is meaningless; that is filtered
# downstream, where it can be seen, not silently here.
texts = df["text"].fillna("").tolist()
n_blank = int((df["text"].str.strip() == "").sum())

print(f"  corpus: {len(df)} rows, {df.response_id.nunique()} participants, "
      f"{n_blank} blank answers")
print(f"  by item: {df.item_type.value_counts().to_dict()}")
# Expect 948 participants and 2,844 rows in a full run: 1,015 raw, 948 after the
# preregistered exclusions of prereg 6.2, three open-ends each. A participant count
# near 1,015 means notebook 00 ran without the exclusions and these vectors will
# not line up with the ensemble scores.

  corpus: 2844 rows, 948 participants, 5 blank answers
  by item: {'ranking': 948, 'backward': 948, 'forward': 948}


In [55]:
# === LOAD 2 of 2: the per-facet reference passages ===
# WHAT THESE ARE: for each of the nine facets, a small set of short passages that
# are unambiguous examples of that facet. A response's facet similarity, computed
# in notebook 09, is its mean cosine to that facet's set. This notebook's only job
# with them is to embed them in the SAME space as the responses.
#
# WHY THEY COME FROM THE SEED HALF ONLY: the corpus is split once, before any
# scoring, into a SEED half and a TEST half (notebook 03, frozen from RANDOM_SEED).
# Reference passages draw only from SEED, so the TEST responses -- the ones any
# convergence number is computed on -- contributed nothing to any reference set.
# Leave-one-out (notebook 09) removes only a response's similarity to ITSELF; it
# cannot remove the shared signal running through the rest of a stratum, so LOO
# alone would leave the convergence number partly circular. The frozen split is
# what closes that, and it is mandatory rather than tidy (MEASUREMENT_DESIGN.md
# section 8, anti-circularity rule 4: never report a convergence number for a
# stream on the responses that seeded that stream). That restriction is enforced
# where the pools are built, in notebook 04, and travels inside this file.
p_seeds = need("embedding_seed_pools.json", produced_by="04")
with open(p_seeds) as f:
    _seeds_json = json.load(f)

# CONTRACT: notebook 04 writes a wrapper object -- the facet mapping under a
# "pools" key, beside the provenance fields recorded with it (random_seed, source)
# -- so the mapping is unwrapped here. A file that is already the bare mapping is
# accepted too, so a hand-built pool file still runs.
if isinstance(_seeds_json, dict) and "pools" in _seeds_json:
    seed_pools = _seeds_json["pools"]
    seed_prov = {k: v for k, v in _seeds_json.items() if k != "pools"}
    # The provenance travels inside the file precisely so this notebook can state
    # where its reference passages came from without trusting a filename.
    print(f"  seed provenance: {seed_prov}")
else:
    seed_pools = _seeds_json
    seed_prov = {}

# Every key must be one of the nine registered facets. This assert is what turns a
# wrong shape into a stop: handed the wrapper object instead of the mapping, every
# key would be skipped below, the seed frame would come back empty, and all nine
# similarity columns in notebook 09 would be NaN -- a failure that reads like a null
# result rather than like a broken load.
_unknown = [k for k in seed_pools if k not in FACETS]
assert not _unknown, f"embedding_seed_pools.json has non-facet keys: {_unknown}"

# A facet with no exemplars in the SEED half legitimately has no pool, so a missing
# facet is announced rather than fatal: it simply carries no embedding measurement,
# and notebook 09 reports it as "not run" rather than as a blank.
_missing = [f for f in FACETS if f not in seed_pools]
if _missing:
    print(f"  [note] no reference pool for: {_missing}")
    print("         those facets will carry no embedding measurement, by design")
print(f"  pools loaded for: {sorted(seed_pools.keys())}")

  [ok]   embedding_seed_pools.json  (written unknown time)
  seed provenance: {'random_seed': 20260629, 'source': 'codebook anchor passages (MEASUREMENT_DESIGN.md section 4) plus word-list-positive responses from outside the TEST half of the frozen split; no ensemble output was consulted; TEST-half responses used: 0; human-coded exemplars used: 0', 'built_by': '04_reference_pools', 'built_at': '2026-08-13T22:09:53', 'run_mode': 'full', 'human_gold': False, 'routes': {'codebook_anchor': 'author-written anchor passages from the codebook, quoted as the codebook prints them, independent of the ensemble by construction', 'wordlist_non_test': 'responses the nine-facet word list scored positive, ranked low on the opposing facets, drawn from everywhere except the TEST half'}, 'routes_that_placed_an_exemplar': ['codebook_anchor', 'wordlist_non_test'], 'settings': {'pool_min': 8, 'pool_max': 15, 'mixed_motive_reserved': 2, 'max_chars': 400, 'min_words': 3, 'anchor_target_min': 0.5}, 'split': {'s

## Do

In [56]:
# === HELPER: L2 row normalization, so a cosine is a plain dot product ===
def normalize_rows(emb_matrix):
    """L2-normalize each row so cosine == dot product.

    Every encoder below is already asked for unit-norm rows
    (normalize_embeddings=True), so on their output this is idempotent -- running
    it twice changes nothing. It is applied anyway, because it makes
    `corpus @ seed.T` a cosine matrix BY CONSTRUCTION rather than by trusting a
    keyword argument in another cell, and because the Voyage API makes no unit-norm
    guarantee at all.

    A cosine here means: the angle between two passages' vectors, 1.0 for the same
    direction and 0.0 for unrelated. Notebook 09 does nothing to these numbers
    except multiply, mask and average them, so if the rows are not unit length the
    whole similarity leg is quietly measuring vector length instead of meaning.
    """
    norms = np.linalg.norm(emb_matrix, axis=1, keepdims=True)
    # Clip the denominator: an all-zero embedding (possible for a degenerate input)
    # would otherwise divide by zero and spray NaN through the matrix.
    return emb_matrix / np.clip(norms, 1e-8, None)

print("[ok] normalize_rows defined")

[ok] normalize_rows defined


In [57]:
# === FLATTEN THE REFERENCE POOLS INTO ONE TABLE ===
# The flat table is what lets one matrix multiply in notebook 09 serve all nine
# facets: the exemplar texts are embedded ONCE here, and the facet masks are
# applied to the resulting matrix. Row order of this table IS the row order of the
# seed block of the matrix this notebook saves, so the two must never be re-sorted
# independently. That is why the table is written into the metadata file: the
# alignment travels with the numbers instead of being reconstructed downstream.
def _seed_frame(seed_pools):
    """seed_pools -> DataFrame(facet, response_id, item_type, route, source, text)."""
    rows = []
    for facet, exemplars in seed_pools.items():
        # There is no "skip this key if it is not a facet" branch here. The load
        # cell above already stops the run on any non-facet key (the _unknown
        # assert), and this function is called once, on that same dict, on the line
        # just below it -- so a skip branch could never run. A check that cannot
        # fire reads like a guarantee and is not one; the guarantee is the assert,
        # and it is in the cell above.
        for ex in exemplars:
            # .get() rather than [] -- a pool entry written without item_type then
            # surfaces as a value matching no response's item_type (the exemplar
            # simply drops out of every set) instead of raising here.
            # TWO PROVENANCE FIELDS, and they are not the same thing. "route" is
            # notebook 04's categorical label for how the exemplar was obtained --
            # "codebook_anchor" for a written anchor passage, "wordlist_seed_half"
            # for a real SEED-half response the word list scored positive -- and it
            # is what notebook 09 tallies in its "by route" line. "source" is the
            # free-text sentence beside it naming the file it came out of. Both are
            # carried, because the category is what a reader counts and the
            # sentence is what they check the count against. A pool without either
            # tag reads as "unknown" and nothing breaks.
            rows.append({"facet": facet,
                         "response_id": ex.get("response_id"),
                         "item_type": ex.get("item_type"),
                         "route": ex.get("route", "unknown"),
                         "source": ex.get("source", "unknown"),
                         "text": ex.get("text", "")})
    # Columns are declared explicitly so that an empty pool still returns a frame
    # carrying a facet column: the count then prints 0, which is legible, where a
    # column-less frame would raise.
    return pd.DataFrame(rows, columns=["facet", "response_id", "item_type",
                                       "route", "source", "text"])

seed_frame = _seed_frame(seed_pools)
print(f"[ok] reference table flattened: {len(seed_frame)} rows")

[ok] reference table flattened: 133 rows


In [58]:
# === IS THAT TABLE FIT TO EMBED? ===
# One row per (facet, response_id, item_type). The SAME response may legitimately
# appear under DIFFERENT facets -- a mixed-motive answer is deliberately placed in
# more than one pool -- so the key checked is the triple and not the response on
# its own. Repeated INSIDE one facet, though, an exemplar would be embedded twice
# and counted twice in that facet's mean cosine, quietly giving one passage double
# the weight of its neighbours, and no shape check anywhere would notice.
_dup_ex = seed_frame.duplicated(["facet", "response_id", "item_type"], keep=False)
assert not _dup_ex.any(), (
    f"embedding_seed_pools.json repeats {int(_dup_ex.sum())} exemplar rows inside "
    f"a facet, for example "
    f"{seed_frame.loc[_dup_ex, ['facet', 'response_id', 'item_type']].head(3).to_dict('records')}"
    f". Each would count twice in that facet's mean cosine. Fix notebook 04.")
# Read the next line as the health check for the whole embedding leg: expect 8 to
# 15 exemplars per facet across 9 facets, so roughly 100 to 140 rows. A count well
# under that means the SEED half barely contains those facets, and every similarity
# computed against that thin pool rests on a handful of passages.
print(f"  seed exemplars: {len(seed_frame)} across "
      f"{seed_frame.facet.nunique()} facets")
if len(seed_frame):
    print(seed_frame.groupby(["facet", "item_type"]).size().to_string())
    # An exemplar whose item_type is not one of the corpus's three would match no
    # response in notebook 09 and would drop out of every reference set without
    # complaint. Say so here, where it can still be fixed in notebook 04.
    stray = sorted(set(seed_frame.item_type.dropna()) - set(df.item_type.unique()))
    if stray:
        print(f"  [note] seed item_type values not in the corpus: {stray}")
        print("         those exemplars will match no response and be ignored")

  seed exemplars: 133 across 9 facets
facet            item_type
censure          backward      2
                 forward      13
desert           backward      6
                 forward       5
                 ranking       4
deterrence       forward       1
                 ranking      14
incapacitation   backward      5
                 forward       3
                 ranking       7
moral_balance    backward      4
                 forward       6
                 ranking       4
proportionality  backward      5
                 forward       9
                 ranking       1
rehabilitation   backward      1
                 forward       5
                 ranking       9
revenge          backward      1
                 forward       7
                 ranking       6
suffering        backward      5
                 forward      10


In [59]:
# === OPEN THE ENCODER ===
# Local checkpoints load once here and are used for BOTH the corpus and the
# reference passages. The API path opens a client instead. Either way, one encoder
# object serves both halves of the file this notebook writes: the passages and the
# responses have to sit in ONE space or the cosine between them is not a similarity.
if ENC["kind"] == "local":
    print(f"Loading {ENC['checkpoint']} ({ENC['size']})...")
    model = SentenceTransformer(ENC["checkpoint"], device=device)
    _width = model.get_sentence_embedding_dimension()
    # THE PIN IS THE PAIR: the checkpoint name AND the width recorded beside it. A
    # checkpoint that loads at some other width is not the model this pipeline is
    # pinned to, whatever the string says, and notebook 09's whole job is comparing
    # encoders whose labels have to be true. Stopping here costs one reload;
    # discovering it after the vectors are written and blessed by done() costs the
    # reader's trust in every number computed from them.
    assert _width == ENC["dim"], (
        f"{ENC['checkpoint']} loaded at width {_width}, but this encoder is pinned "
        f"at {ENC['dim']} in ENCODERS above. Either the checkpoint has moved or the "
        f"pinned width is wrong. Fix one of them before anything is encoded.")
    print(f"[ok] loaded on {device}, width {_width} as pinned")
else:
    # WHY COLAB SECRETS RATHER THAN A KEY TYPED INTO A CELL: userdata.get() returns
    # the value at runtime and stores nothing in the notebook file, so this .ipynb
    # can go to OSF with the rest of the analysis code (prereg 6.7) without a key
    # baked into it.
    # SETUP: Colab left sidebar > key icon > add a secret named exactly
    # "Punishment_VoyageAPI", then switch notebook access ON for this file.
    from google.colab import userdata
    _key = userdata.get("Punishment_VoyageAPI")
    # userdata.get returns None or "" when the secret exists but notebook access is
    # off. A silently empty key would not fail here -- it would fail minutes later,
    # mid-encode, as an opaque auth error inside the retry loop. Turn it into an
    # exception now, while the cause is still on screen. Voyage is optional as a
    # MODEL (set ENCODER_KEY back to "bge" and this notebook needs no key at all),
    # but a half-configured key is a setup fault, not something to work around.
    if not _key:
        raise ValueError(
            "Punishment_VoyageAPI is empty or notebook access is off. Fix the "
            "secret, or set ENCODER_KEY = 'bge' and run the required leg instead.")
    # Last four characters only: enough to confirm WHICH key loaded, not enough to
    # leak it through a saved notebook's stored output.
    print(f"[ok] Voyage key loaded (ends in ...{_key[-4:]})")
    voyage_client = voyageai.Client(api_key=_key)
    print(f"[ok] Voyage client ready (model = {ENC['checkpoint']})")

[ok] Voyage key loaded (ends in ...weWc)
[ok] Voyage client ready (model = voyage-3-large)


In [60]:
# === HOW A LOCAL CHECKPOINT TURNS TEXT INTO VECTORS ===
if ENC["kind"] == "local":
    def embed(sentences, progress=False, batch_size=None):
        """Encode a list of strings into one row-normalized matrix."""
        # batch_size defaults to this encoder's pinned setting -- 32, the Dark Side
        # setting, kept so the two pipelines' encoder runs stay comparable, and it
        # fits a T4 alongside a 1.3 GB checkpoint without tuning. It can be
        # overridden per call, so that this function and the Voyage one below take
        # the same arguments and every call site can be written once for both.
        # Batch size changes throughput only -- the vectors are identical.
        # normalize_embeddings=True is what makes the saved matrix usable directly
        # as one side of notebook 09's dot product.
        # convert_to_numpy=True returns a plain array rather than a torch tensor,
        # so nothing downstream needs a GPU to read this file.
        # The two encode paths are called with the SAME flags on purpose: they must
        # differ only in the checkpoint, so that a difference in notebook 09's
        # sensitivity table is attributable to the encoder rather than to the call.
        return model.encode(list(sentences),
                            batch_size=batch_size or ENC["batch_size"],
                            show_progress_bar=progress,
                            convert_to_numpy=True,
                            normalize_embeddings=True)
    print(f"[ok] local encode path ready (batch_size={ENC['batch_size']})")
else:
    print("[skip] local encode path not used on the Voyage run")

[skip] local encode path not used on the Voyage run


In [61]:
# === HOW THE VOYAGE API TURNS TEXT INTO VECTORS (batches, retries, blanks) ===
if ENC["kind"] == "api":
    def embed(sentences, progress=False, batch_size=None):
        """Encode through the Voyage API: batched, retried, positionally aligned."""
        text_list = list(sentences)
        batch_size = batch_size or ENC["batch_size"]
        out = []
        # batch_size=64 for the corpus: large enough that ~2,800 responses cost
        # about 45 round trips instead of 2,800, small enough to stay inside the
        # API's per-request token ceiling for open-ends of this length.
        for i in range(0, len(text_list), batch_size):
            # BLANK GUARD. The API rejects an empty string outright and would fail
            # the entire batch, taking 63 good texts with it. A single space is
            # accepted and yields a meaningless vector -- which is correct, because
            # the row it belongs to is a blank open-end. SUBSTITUTING rather than
            # dropping is the point: `out` stays positionally aligned to text_list,
            # which is what every row-count check downstream is checking.
            batch = [t if isinstance(t, str) and t.strip() else " "
                     for t in text_list[i:i + batch_size]]
            for attempt in range(3):
                try:
                    # input_type=None is deliberate. Voyage's "query" and
                    # "document" modes prepend different instructions and place the
                    # two sides in asymmetric spaces. Responses and reference
                    # passages must be embedded IDENTICALLY, or the cosine between
                    # them is not a similarity within one space and the whole leg
                    # is meaningless.
                    result = voyage_client.embed(batch, model=ENC["checkpoint"],
                                                 input_type=None)
                    out.extend(result.embeddings)
                    break
                except Exception as e:
                    # Third failure re-raises rather than continuing: better to stop
                    # the run than to save a short or gap-ridden matrix that some
                    # future reload would then bless.
                    if attempt == 2:
                        raise
                    # Exponential backoff, 1s then 2s -- enough for a rate-limit
                    # window to clear without stalling a long run.
                    wait = 2 ** attempt
                    print(f"  retry {attempt + 1} after {wait}s: {e}")
                    time.sleep(wait)
            if progress:
                print(f"  ...{min(i + batch_size, len(text_list))}/{len(text_list)}")
        # The API makes no unit-norm guarantee, so normalize on the way out.
        return normalize_rows(np.array(out))
    print(f"[ok] Voyage encode path ready (batch_size={ENC['batch_size']})")
else:
    print("[skip] Voyage encode path not used on a local run")

[ok] Voyage encode path ready (batch_size=64)


In [62]:
# === THE RESPONSE VECTORS: reuse the saved matrix if it still fits, else encode ===
# WHY A CACHE AT ALL: a Colab VM is ephemeral, and a full encode costs minutes on a
# T4, far longer on CPU, and real money on the API. Re-running this notebook to add
# an optional encoder, or after a crash, should not re-encode what is already done.
# WHAT IS REUSED: the answers only. The reference passages are re-embedded on every
# run, for the reason set out in the cell after this one.
# WHY THE CHECKS: a .npy file carries no index. A matrix left over from a different
# corpus -- a pilot export, a differently filtered openends_long.csv, a smoke run --
# would still multiply cleanly against the reference passages and return
# plausible-looking similarities silently attached to the wrong rows. The recorded
# row keys, row counts and width are the only alignment checks there are. Where a
# check fails we throw the CACHE away and re-encode; we never reshape or re-filter
# the corpus to make a cache fit. df is the corpus, the saved matrix is the
# disposable artefact.
corpus_keys = [f"{r}|{i}" for r, i in
               zip(df.response_id.astype(str), df.item_type.astype(str))]
corpus_emb = None

if os.path.exists(OUT_NPY) and os.path.exists(OUT_META):
    with open(OUT_META) as f:
        _old = json.load(f)
    _cached = np.load(OUT_NPY)
    _n_c = int(_old.get("n_corpus", -1))
    why = []
    if _old.get("checkpoint") != ENC["checkpoint"]:
        why.append(f"written with {_old.get('checkpoint')}")
    if _old.get("run_mode") != RUN_MODE:
        why.append(f"written in RUN_MODE={_old.get('run_mode')}")
    if _cached.shape[0] != _n_c:
        why.append(f"{_cached.shape[0]} rows against the {_n_c} its own metadata "
                   f"describes")
    elif _cached.shape[1] != ENC["dim"]:
        # A matrix of the right height and the wrong width is a DIFFERENT encoder's
        # output sitting under this encoder's name. Nothing else would notice: it
        # multiplies cleanly and returns numbers.
        why.append(f"width {_cached.shape[1]} against the pinned {ENC['dim']}")
    elif _cached.shape[0] != len(df):
        why.append(f"{_cached.shape[0]} saved answers against {len(df)} corpus rows")
    elif _old.get("corpus_keys") != corpus_keys:
        why.append("different response_id/item_type rows or row order")
    if why:
        print("  saved matrix set aside (" + "; ".join(why) + ")")
        if ENC["kind"] == "api":
            print("  note: re-encoding through the API costs API spend")
    else:
        corpus_emb = _cached
        print(f"  reusing the saved response vectors: {corpus_emb.shape}")

if corpus_emb is None:
    print("  no reusable saved vectors: encoding from scratch in the next cell.")

  no reusable saved vectors: encoding from scratch in the next cell.


In [63]:
# === THE RESPONSE VECTORS, PART 2: encode whatever the cache could not supply ===
if corpus_emb is None:
    print(f"Encoding {len(texts)} responses with {ENC['checkpoint']}...")
    t0 = time.time()
    corpus_emb = normalize_rows(embed(texts, progress=True))
    print(f"  encoded in {time.time() - t0:.1f} sec")

# One row per answer. On the API path a batch whose reply came back short would
# arrive here as a matrix with fewer rows than texts, and every row after the gap
# would belong to the wrong person.
assert corpus_emb.shape[0] == len(df), (
    f"the response matrix has {corpus_emb.shape[0]} rows for {len(df)} answers")
# THE PIN IS THE PAIR: checkpoint AND width, checked on the vectors themselves and
# checked BEFORE anything is written. On the API path this is the first moment the
# real width is knowable at all, since no checkpoint is loaded locally to ask. A
# wrong width means either the checkpoint moved or the width recorded in ENCODERS
# is wrong; either way notebook 09 would be comparing encoders whose labels lie,
# and done() would already have blessed the file by the time anyone noticed.
assert corpus_emb.shape[1] == ENC["dim"], (
    f"{ENC['checkpoint']} returned width {corpus_emb.shape[1]}, but this encoder "
    f"is pinned at {ENC['dim']}. Fix the pin or the checkpoint; nothing is written "
    f"until they agree.")
print(f"[ok] response vectors: {corpus_emb.shape}, width as pinned")

Encoding 2844 responses with voyage-3-large...
  ...64/2844
  ...128/2844
  ...192/2844
  ...256/2844
  ...320/2844
  ...384/2844
  ...448/2844
  ...512/2844
  ...576/2844
  ...640/2844
  ...704/2844
  ...768/2844
  ...832/2844
  ...896/2844
  ...960/2844
  ...1024/2844
  ...1088/2844
  ...1152/2844
  ...1216/2844
  ...1280/2844
  ...1344/2844
  ...1408/2844
  ...1472/2844
  ...1536/2844
  ...1600/2844
  ...1664/2844
  ...1728/2844
  ...1792/2844
  ...1856/2844
  ...1920/2844
  ...1984/2844
  ...2048/2844
  ...2112/2844
  ...2176/2844
  ...2240/2844
  ...2304/2844
  ...2368/2844
  ...2432/2844
  ...2496/2844
  ...2560/2844
  ...2624/2844
  ...2688/2844
  ...2752/2844
  ...2816/2844
  ...2844/2844
  encoded in 55.2 sec
[ok] response vectors: (2844, 1024), width as pinned


In [64]:
# === THE REFERENCE PASSAGES: embedded fresh on every single run ===
# WHY THE SAME CHECKPOINT AS THE RESPONSES: the reference passages and the
# responses have to live in ONE space, or the cosine between them is not a
# similarity at all. They are embedded here, with the encoder object that just
# embedded the corpus, and are never carried over from another encoder's run.
# WHY THEY ARE NEVER RELOADED FROM DISK: there are only about 120 of them and they
# cost seconds, and a stale set of reference vectors is much harder to notice than
# a stale set of response vectors -- no row-count check could catch it, because the
# pool size legitimately changes whenever notebook 04 rebuilds the pools. So they
# are re-embedded on every run, including the runs where the response vectors came
# back from the saved matrix, and the file written below is an output for notebook
# 09 rather than a cache this notebook ever reads. That is the one place in this
# notebook where we spend time on purpose.
# fillna(" ") -- a single space rather than "" -- keeps the encoders and the Voyage
# API happy on a malformed pool entry. A blank exemplar would be an upstream fault,
# not something to score, and it shows up as an uninformative passage rather than
# as a crash.
if len(seed_frame):
    # batch_size=32 for the roughly 120 reference passages, even where the corpus
    # went through at 64 on the Voyage path: it is one or two calls either way, and
    # the smaller batch keeps a single transient API failure cheap to retry. On a
    # local checkpoint the batch size changes throughput only -- the vectors are
    # identical -- so both encode paths take the argument and this one call site
    # serves both.
    seed_emb = normalize_rows(embed(seed_frame["text"].fillna(" ").tolist(),
                                    batch_size=32))
else:
    # An empty pool file is a real state, not an error: it means notebook 04 found
    # no admissible exemplars. Write an empty matrix and let notebook 09 report the
    # embedding leg as "not run" rather than as a failure.
    seed_emb = np.zeros((0, corpus_emb.shape[1]), dtype=corpus_emb.dtype)
    print("  [note] no reference passages at all, the reference matrix is empty")

# Row order of seed_frame IS row order here: the facet labels written into the
# metadata below describe THESE rows, in THIS order, and nothing may re-sort either
# side independently.
assert seed_emb.shape[0] == len(seed_frame), \
    "reference matrix is not one row per exemplar"
# Both matrices must describe ONE space. Notebook 09 checks this again on the two
# files it reads, and it is checked here as well, because here is where the two
# widths can still be traced back to the encode that produced them: different
# widths mean the passages and the answers are not in the same space, so no cosine
# between them is a similarity.
assert seed_emb.shape[1] == corpus_emb.shape[1], \
    "responses and references have different widths: they are not the same space"
print(f"[ok] reference vectors: {seed_emb.shape}, "
      f"embedded with {ENC['checkpoint']}")

[ok] reference vectors: (133, 1024), embedded with voyage-3-large


## Save and check

In [65]:
# === WRITE: the response vectors and the reference vectors, always together ===
# THE TWO MATRICES ARE ALWAYS WRITTEN IN THE SAME RUN, BY THE SAME CHECKPOINT.
# Notebook 09 multiplies one by the other, so a fresh set of responses paired with
# somebody else's stale reference passages is the failure mode no check downstream
# could catch: a stale reference matrix has the right shape and returns perfectly
# plausible numbers. Writing both here, every time, is what prevents it.
# WHY THE WRITE IS THE LAST THING THAT HAPPENS: every encode above has to have
# returned before a single byte is saved, so a run that dies mid-encode leaves no
# half-built matrix behind for a later run to reload and bless.
# float32 halves the files and moves a cosine in about the seventh decimal place.

# ONE ROW PER (response_id, item_type), checked on what is actually being written
# rather than on the frame it was built from. corpus_keys IS the index notebook 09
# matches row for row against openends_long.csv: a repeated key there would hand
# two identical similarity rows to one person, and every n downstream would read
# high while the printed participant total still said 948.
assert len(set(corpus_keys)) == len(corpus_keys), (
    f"{len(corpus_keys) - len(set(corpus_keys))} duplicated response_id|item_type "
    f"keys are about to be written beside these vectors. Every row of the response "
    f"matrix has to name a different answer.")
assert len(corpus_keys) == corpus_emb.shape[0], (
    f"{len(corpus_keys)} row keys for {corpus_emb.shape[0]} rows of vectors: the "
    f"index and the numbers are describing different things.")

corpus_emb = corpus_emb.astype("float32")
seed_emb = seed_emb.astype("float32")
np.save(OUT_NPY, corpus_emb)
np.save(OUT_SEED, seed_emb)
print(f"  saved: {corpus_emb.shape[0]} answers and {seed_emb.shape[0]} reference "
      f"passages, both {corpus_emb.shape[1]} numbers wide")

  saved: 2844 answers and 133 reference passages, both 1024 numbers wide


In [66]:
# === AND THE INDEX THAT SAYS WHAT EVERY ROW OF EACH ONE IS ===
# The metadata file is what makes the two matrices readable by anything other than
# this notebook: how wide the vectors are, how many rows are in each, which answer
# each row of the response matrix belongs to, and which facet each row of the
# reference matrix belongs to. Without it a .npy is an anonymous pile of numbers,
# and notebook 09 skips any encoder that arrives without one.
meta = {
    "encoder_key": ENCODER_KEY,
    "maker": ENC["maker"],
    "checkpoint": ENC["checkpoint"],          # the pin, for the OSF record
    "kind": ENC["kind"],
    "device": device,
    "dim": int(corpus_emb.shape[1]),
    "batch_size": ENC["batch_size"],
    "row_normalized": True,                   # so a dot product IS a cosine
    # HOW MANY ROWS IN EACH FILE. Notebook 09 checks both matrices against these
    # two numbers before it reads a value out of either.
    "n_corpus": int(corpus_emb.shape[0]),     # rows of embeddings_<key>.npy
    "n_seed": int(seed_emb.shape[0]),         # rows of embeddings_<key>_seed.npy
    # "response_id|item_type" for every row of the response matrix, in row order.
    # This is the alignment contract: it lets notebook 09 check the matrix against
    # openends_long.csv rather than assume two files happen to be in the same order.
    "corpus_keys": corpus_keys,
    # facet, response_id, item_type and both provenance fields for every row of the
    # reference matrix, in row order. Carrying the facet labels here is what lets a
    # reader slice that matrix per facet without rebuilding the pools and risking a
    # different order; carrying route is what lets notebook 09 report how many
    # passages came in by each route without re-opening the pool file.
    "seed_rows": seed_frame[["facet", "response_id", "item_type",
                             "route", "source"]].to_dict("records"),
    "seed_pool_provenance": seed_prov,
    "run_mode": RUN_MODE,
    "random_seed": RANDOM_SEED,
    "written_by": f"{NB_ID}_{NB_NAME}",
}
with open(OUT_META, "w") as f:
    # default=str so an unusual id type coming out of pandas (a numpy integer, for
    # instance) is written as text instead of stopping the write. Every field here
    # is descriptive rather than arithmetic, so text is a safe fallback.
    json.dump(meta, f, indent=1, default=str)

done(OUT_NPY, rows=meta["n_corpus"],
     note=f"{ENCODER_KEY}: one row per answer, in openends_long.csv order, "
          f"width {meta['dim']}, L2-normalized")
done(OUT_SEED, rows=meta["n_seed"],
     note=f"{ENCODER_KEY}: one row per reference passage, same checkpoint, "
          f"same run, re-embedded every time")
done(OUT_META, rows=1,
     note=f"row keys, facet labels, the row counts and the pinned checkpoint "
          f"for {OUT_NPY} and {OUT_SEED}")

  [wrote] embeddings_voyage.npy  (2844 rows, 11,649,152 bytes)
  [wrote] embeddings_voyage_seed.npy  (133 rows, 544,896 bytes)
  [wrote] embeddings_voyage_meta.json  (1 rows, 122,658 bytes)


In [46]:
# === WHAT CAME OUT, AND WHAT TO RUN NEXT ===
# A last look at the numbers themselves, because a matrix of the right shape can
# still be wrong. Row lengths must all be 1.0: that is what normalization means,
# and it is what makes notebook 09's dot product a cosine rather than a product of
# two vector lengths. The width was checked against the pin before anything was
# written, in the load and encode cells above, so it is reported here and not
# re-tested.
_norms = np.linalg.norm(np.vstack([corpus_emb, seed_emb]), axis=1)
print(f"\nNOTEBOOK {NB_ID} SUMMARY -- encoder {ENCODER_KEY}")
print(f"  checkpoint      : {ENC['checkpoint']}")
print(f"  responses       : {meta['n_corpus']} rows x {meta['dim']} numbers")
print(f"  references      : {meta['n_seed']} rows x {meta['dim']} numbers")
print(f"  width           : {meta['dim']}, the width this checkpoint is pinned at")
print(f"  row lengths     : min {_norms.min():.4f}, max {_norms.max():.4f} "
      f"(both should read 1.0000)")
print(f"  blank answers   : {n_blank} of {len(df)} rows, embedded but meaningless")

# Which of the three legs exist in this folder so far, so the reader can see at a
# glance what is left to run rather than working it out from filenames.
print("\n  encoders run in this folder:")
for k, v in ENCODERS.items():
    # ALL THREE files, because notebook 09 needs all three. Response vectors on
    # their own have nothing to take a cosine against, and a matrix arriving with
    # no metadata beside it is an anonymous pile of numbers that notebook 09 skips.
    # An encoder with a partial set is "not run".
    ran = all(os.path.exists(f) for f in (f"embeddings_{k}.npy",
                                          f"embeddings_{k}_seed.npy",
                                          f"embeddings_{k}_meta.json"))
    role = "REQUIRED" if v["required"] else "optional"
    print(f"    {k:7s} {role:9s} {'done' if ran else 'not run'}")

print("\nNEXT:")
if not all(os.path.exists(f) for f in ("embeddings_bge.npy",
                                       "embeddings_bge_seed.npy",
                                       "embeddings_bge_meta.json")):
    print("  1. Set ENCODER_KEY = 'bge' and run this notebook again. BGE is the")
    print("     one encoder the pipeline requires.")
else:
    print("  1. Optionally set ENCODER_KEY to 'mpnet' or 'voyage' and run this")
    print("     notebook again. Each optional encoder adds a column to the")
    print("     model-sensitivity report; none of them is required.")
print("  2. Notebook 09 turns these vectors into per-facet similarity scores:")
print("     mean cosine to each facet's reference set, per item_type, with")
print("     leave-one-out. Nothing here can pass or fail a facet.")


NOTEBOOK 08 SUMMARY -- encoder mpnet
  checkpoint      : sentence-transformers/all-mpnet-base-v2
  responses       : 2844 rows x 768 numbers
  references      : 133 rows x 768 numbers
  width           : 768, the width this checkpoint is pinned at
  row lengths     : min 1.0000, max 1.0000 (both should read 1.0000)
  blank answers   : 5 of 2844 rows, embedded but meaningless

  encoders run in this folder:
    bge     REQUIRED  done
    mpnet   optional  done
    voyage  optional  not run

NEXT:
  1. Optionally set ENCODER_KEY to 'mpnet' or 'voyage' and run this
     notebook again. Each optional encoder adds a column to the
     model-sensitivity report; none of them is required.
  2. Notebook 09 turns these vectors into per-facet similarity scores:
     mean cosine to each facet's reference set, per item_type, with
     leave-one-out. Nothing here can pass or fail a facet.
